# Week 2, Day 2 — LangChain
### Tools, Chains, Memory & Your First Framework Agent

Yesterday's agent was built by hand. Today it's rebuilt with LangChain, on
top of the Gemini API (free tier — same reason as Day 1). The goal isn't
"LangChain is better" — it's seeing exactly what LangChain automates for
you, and where that automation hides things you had full visibility into
yesterday.

**Setup:**
```
pip install langchain langchain-classic langchain-google-genai
```
`langchain-classic` is needed because LangChain 1.x moved its main agent
API to the newer `create_agent` (LangGraph-based) — the classic
`create_tool_calling_agent` + `AgentExecutor` 

In [1]:
import os

os.environ["GEMINI_API_KEY"] = ""

if not os.environ.get("GEMINI_API_KEY"):
    print("⚠️  GEMINI_API_KEY not set yet. Set it above before running the later cells.")
else:
    print("API key found — ready to go.")

API key found — ready to go.


## Task 1 — LangChain Setup & Core Concepts

### Mapping to Day 1's raw-Python equivalents

| LangChain concept | Day 1 raw-Python equivalent |
|---|---|
| `ChatGoogleGenerativeAI` (LLM wrapper) | `genai.Client()` — the raw API client |
| `@tool` decorator | Our `TOOLS` schema list + the plain Python function, combined into one object |
| `create_tool_calling_agent` + `AgentExecutor` | Our `run_agent()` while-loop (Reason → Act → Observe → repeat) |
| Memory (`RunnableWithMessageHistory`) | Our `history` list, manually resent every call |

Nothing here is a new *concept* — Day 1 already built all four of these by
hand. LangChain gives each one a named, reusable class instead of
project-specific code.

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0.3)

# --- A basic LCEL pipeline: prompt -> llm -> output parser ---
basic_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise assistant."),
    ("human", "{question}"),
])

basic_chain = basic_prompt | llm | StrOutputParser()

result = basic_chain.invoke({"question": "In one sentence, what is a ReAct agent?"})
print(result)

C:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


A ReAct agent is an AI system that interleaves reasoning (thinking about a task) and acting (using tools or taking actions) to solve complex problems step-by-step.


**What the `|` (pipe) syntax is doing under the hood:** each LCEL
component (`prompt`, `llm`, `StrOutputParser()`) implements the same
`Runnable` interface (`.invoke()`, `.stream()`, `.batch()`). The `|`
operator is Python's overloaded `__or__`, which LangChain uses to wrap two
Runnables into a `RunnableSequence` — calling `.invoke()` on the sequence
just calls `.invoke()` on the first component and feeds its output as the
input to the next, left to right. It's functionally the same as writing
`StrOutputParser().invoke(llm.invoke(prompt.invoke({"question": ...})))`,
just with a much more readable syntax and automatic support for
streaming/batching across the whole chain.

## Task 2 — Define & Register Tools

Three tools via the `@tool` decorator: `calculator` and `get_weather`
reused from Day 1, plus a new one — `get_product_price` — that reads from
a real local JSON "database" (`products.json`), created below.

**Why tool docstrings matter (same principle as Day 1's `description`
field):** LangChain's `@tool` decorator reads the function's docstring and
uses it verbatim as the tool's `description` sent to the model — the
docstring effectively becomes part of the prompt on every call. The model
never inspects the function body, so an unclear or missing docstring means
the model is guessing blind about when to call the tool and what to pass
it. This is the exact same mechanism as Day 1's JSON schema `description`
field, just auto-generated from Python instead of written by hand.

In [3]:
import json
import ast
import operator as op
from langchain_core.tools import tool, ToolException

# --- Create a real local JSON "database" for the new tool to read from ---
products_db = {
    "iphone 15": {"price_usd": 799, "category": "smartphone", "brand": "Apple"},
    "samsung galaxy s24": {"price_usd": 699, "category": "smartphone", "brand": "Samsung"},
    "google pixel 8": {"price_usd": 599, "category": "smartphone", "brand": "Google"},
}
with open("products.json", "w") as f:
    json.dump(products_db, f, indent=2)

print("products.json written:", list(products_db.keys()))

products.json written: ['iphone 15', 'samsung galaxy s24', 'google pixel 8']


In [9]:
_ALLOWED_OPS = {
    ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv,
    ast.Pow: op.pow, ast.USub: op.neg,
}

def _safe_eval(node):
    if isinstance(node, ast.Constant):
        if isinstance(node.value, (int, float)):
            return node.value
        raise ValueError("Only numeric constants are allowed")
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError(f"Unsupported expression element: {node!r}")


@tool
def calculator(expression: str) -> str:
    """Evaluates a single arithmetic expression using +, -, *, /, **, and
    parentheses, e.g. '12 * (4 + 1)'. Use this for any numeric computation.
    Does not understand word problems or units — convert to a plain
    numeric expression first."""
    tree = ast.parse(expression, mode="eval")
    return str(_safe_eval(tree.body))


_FAKE_WEATHER = {
    "lahore": {"temp_c": 34, "condition": "Sunny"},
    "karachi": {"temp_c": 29, "condition": "Humid"},
    "islamabad": {"temp_c": 27, "condition": "Partly cloudy"},
}

@tool
def get_weather(city: str) -> str:
    """Looks up the current weather for a named city and returns
    temperature in Celsius and a short condition string. Stub/demo data
    only — not a live weather feed. City must be a real city name."""
    key = city.strip().lower()
    if key not in _FAKE_WEATHER:
        raise ToolException(f"No weather data available for '{city}' (stub only covers a fixed city list)")
    return json.dumps({"city": city, **_FAKE_WEATHER[key]})


@tool
def get_product_price(product_name: str) -> str:
    """Looks up a product's price and details from the local product
    database (products.json). Use this whenever the user asks about the
    price, brand, or category of a specific product, e.g. 'iPhone 15'.
    Raises an error if the product is not found in the database."""
    with open("products.json") as f:
        db = json.load(f)
    key = product_name.strip().lower()
    if key not in db:
        raise ToolException(f"'{product_name}' not found in product database. Available: {list(db.keys())}")
    return json.dumps({"product": product_name, **db[key]})


tools = [calculator, get_weather, get_product_price]
for t in tools:
    t.handle_tool_error = True
print("Tools registered:", [t.name for t in tools])
print("\nExample auto-generated schema (get_product_price):")
print(get_product_price.args)

Tools registered: ['calculator', 'get_weather', 'get_product_price']

Example auto-generated schema (get_product_price):
{'product_name': {'title': 'Product Name', 'type': 'string'}}


## Task 3 — Build an Agent with `create_tool_calling_agent` / `AgentExecutor`

`create_tool_calling_agent` wires the LLM, tools, and prompt together into
an agent; `AgentExecutor` is what actually *runs* it in a loop — it is,
almost line for line, LangChain's version of Day 1's `run_agent()`
function. `verbose=True` prints the same kind of reasoning/action/
observation trace we hand-wrote with `log()` calls yesterday.

In [10]:
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import MessagesPlaceholder

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a careful assistant with access to tools. Reason step "
               "by step. Only call a tool when actually needed."),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=agent_prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,          # <- this is what prints the reasoning trace below
    handle_tool_error=True,  # Task 5: turn a ToolException into an observation instead of a crash
    max_iterations=6,        # same guardrail idea as Day 1's max_iterations
)

# Multi-step query: needs get_product_price AND calculator
trace_result = agent_executor.invoke({
    "input": "What is the price of the iPhone 15, and what would that price be with a 10% discount?"
})
print("\nFINAL ANSWER:", trace_result["output"])



> Entering new AgentExecutor chain...


C:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Invoking: `get_product_price` with `{'product_name': 'iPhone 15'}`


{"product": "iPhone 15", "price_usd": 799, "category": "smartphone", "brand": "Apple"}

C:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Invoking: `calculator` with `{'expression': '799 * 0.9'}`


719.1

C:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'The original price of the iPhone 15 is **$799.00**. \n\nWith a 10% discount, the price would be **$719.10**.', 'index': 0, 'extras': {'signature': 'El4KXAERTTIPEklxnVILDQTcjFjfgzLvB+OvtmbAI1L5B4A5JBBhVb71Jz6XS8q5VXi7Yod2ifhIS+EjsgYRM+mDUK13mKJkmywDpm685jaxIAiMk9Y1GPN5hlNCy8is'}}]

> Finished chain.

FINAL ANSWER: [{'type': 'text', 'text': 'The original price of the iPhone 15 is **$799.00**. \n\nWith a 10% discount, the price would be **$719.10**.', 'index': 0, 'extras': {'signature': 'El4KXAERTTIPEklxnVILDQTcjFjfgzLvB+OvtmbAI1L5B4A5JBBhVb71Jz6XS8q5VXi7Yod2ifhIS+EjsgYRM+mDUK13mKJkmywDpm685jaxIAiMk9Y1GPN5hlNCy8is'}}]


### Annotate the trace above

Once you've run the cell, go through the printed trace (the `verbose=True`
output) and mark, in your own words, where each ReAct stage happened:

- **Reason** — the model's plain-text thought before deciding to act (in
  `AgentExecutor`'s trace this is the `Invoking:` / assistant reasoning
  shown just before a tool call).
- **Act** — each `Invoking: get_product_price` / `Invoking: calculator`
  line — the tool name and the exact arguments passed.
- **Observe** — the line right after, showing what the tool returned.

**Compare to Day 1's raw log:**
- **Similar:** the underlying loop is identical — reason, act, observe,
  repeat until a final answer. The *information* in the trace (which tool,
  what arguments, what came back) is the same information Day 1's
  `[REASON]/[ACT]/[OBSERVE]` prints showed.
- **What's hidden now:** the actual loop code (the `while`/`for` that
  checks for tool calls and re-invokes the model) is inside
  `AgentExecutor` — you no longer write or see it. The exact prompt
  LangChain assembles behind `agent_scratchpad` (how prior tool calls and
  results get serialized back into the conversation) is also invisible
  unless you go looking for it. Day 1 made every one of these steps an
  explicit, readable line of Python; LangChain trades that visibility for
  less code to write.

## Task 4 — Add Memory

The textbook way to do this is `RunnableWithMessageHistory`, which wraps
`agent_executor` so it automatically loads/appends a conversation history
per `session_id` — the same role Day 1's `history` list played, except
LangChain manages the read/append/resend cycle for you.

**In practice, this hit a real bug**, worth documenting rather than
hiding: `RunnableWithMessageHistory` stores the AI's raw output message
verbatim, and Gemini's newer "thinking" models attach an extra signed
metadata block (`extras.signature`) to that message's content. On the next
turn, when LangChain tries to replay that stored message back into the
prompt's `MessagesPlaceholder("chat_history")`, its message-coercion utility
doesn't know how to reconstruct a valid message from that raw content
block, and the whole chain crashes with a `KeyError`/`ValueError`
(`MESSAGE_COERCION_FAILURE`) on turn 2 — one call *after* the automatic
history looked like it worked.

**The fix:** manage memory manually — the same idea `ConversationBufferMemory`
is built on (a plain list of messages), except we explicitly re-wrap each
final answer as a clean, plain-text `AIMessage` before storing it, which
strips Gemini's internal signature metadata and guarantees the next turn's
prompt can always reconstruct it. This is functionally identical to Day 1's
`history.append(...)` pattern.

In [11]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []  # plays the same role as ConversationBufferMemory's internal buffer

def ask_with_memory(user_input):
    result = agent_executor.invoke({"input": user_input, "chat_history": chat_history})
    # Store a clean, plain-text AIMessage — NOT the raw Gemini message object,
    # which can carry "thinking" signature blocks that break replay (see note above).
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=result["output"]))
    return result["output"]

# Turn 1
print("TURN 1:", ask_with_memory("Find the price of the iPhone 15."))
print()
# Turn 2 — depends on turn 1's context
print("TURN 2:", ask_with_memory("Now compare it to the Samsung Galaxy S24."))
print()
# Turn 3 — depends on both prior turns
turn3_answer = ask_with_memory("Which one should I recommend to a budget-conscious client?")
print("TURN 3:", turn3_answer)



> Entering new AgentExecutor chain...


C:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Invoking: `get_product_price` with `{'product_name': 'iPhone 15'}`


{"product": "iPhone 15", "price_usd": 799, "category": "smartphone", "brand": "Apple"}

C:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'The iPhone 15 is priced at $799.', 'index': 0, 'extras': {'signature': 'El4KXAERTTIPYCUvUU+Gd7rAHiRhjIUx6kulWSA3SuVW5erwtKVvFZfSjma0BNsEICGHu4609F7rBrRLbLe+dixfVISWvTt1L4jAvQu6WDTB7IU0yRNOdIuRs/5mMOZH'}}]

> Finished chain.
TURN 1: [{'type': 'text', 'text': 'The iPhone 15 is priced at $799.', 'index': 0, 'extras': {'signature': 'El4KXAERTTIPYCUvUU+Gd7rAHiRhjIUx6kulWSA3SuVW5erwtKVvFZfSjma0BNsEICGHu4609F7rBrRLbLe+dixfVISWvTt1L4jAvQu6WDTB7IU0yRNOdIuRs/5mMOZH'}}]



> Entering new AgentExecutor chain...


...


Invoking: `get_product_price` with `{'product_name': 'Samsung Galaxy S24'}`


{"product": "Samsung Galaxy S24", "price_usd": 699, "category": "smartphone", "brand": "Samsung"}

C:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Here is a comparison between the iPhone 15 and the Samsung Galaxy S24 based on price and details from the database:\n\n* **iPhone 15:** $799 (Brand: Apple, Category: smartphone)\n* **Samsung Galaxy S24:** $699 (Brand: Samsung, Category: smartphone)\n\nThe Samsung Galaxy S24 is $100 cheaper than the iPhone 15.', 'index': 0, 'extras': {'signature': 'El4KXAERTTIPOEkroB/BBlK6Cwu6NH2BDBnwLEkv48aKkL7H+n/4FVs63VoGjwkvCiCHSpvNmPTKkx0hzsEoH04Hwvw6OgO2rCJFSMI6E52cbjcG8aOYcHlYjijFLnoZ'}}]

> Finished chain.
TURN 2: [{'type': 'text', 'text': 'Here is a comparison between the iPhone 15 and the Samsung Galaxy S24 based on price and details from the database:\n\n* **iPhone 15:** $799 (Brand: Apple, Category: smartphone)\n* **Samsung Galaxy S24:** $699 (Brand: Samsung, Category: smartphone)\n\nThe Samsung Galaxy S24 is $100 cheaper than the iPhone 15.', 'index': 0, 'extras': {'signature': 'El4KXAERTTIPOEkroB/BBlK6Cwu6NH2BDBnwLEkv48aKkL7H+n/4FVs63VoGjwkvCiCHSpvNmPTKkx0hzsEoH0

C:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'For a budget-conscious client, the **Samsung Galaxy S24** would be the better recommendation. \n\nHere is why:\n* **Lower Price:** At **$699**, it is $100 less expensive than the iPhone 15 (**$799**).\n* **Value:** Both are flagship smartphones in the same general tier, so saving $100 while still getting a top-tier device makes the Galaxy S24 the more economical choice.', 'index': 0, 'extras': {'signature': 'El4KXAERTTIPn66V3WAbSvh28dPfJ5z6+8ifn7wkjXt66nZ21XCTqCXxBfHkr3/X/iRTMqljXG0SQBZgLZSBWAbuZQWN083sbL6Xjfeg3vvsK+EdeEJptA+9jc0PKMRr'}}]

> Finished chain.
TURN 3: [{'type': 'text', 'text': 'For a budget-conscious client, the **Samsung Galaxy S24** would be the better recommendation. \n\nHere is why:\n* **Lower Price:** At **$699**, it is $100 less expensive than the iPhone 15 (**$799**).\n* **Value:** Both are flagship smartphones in the same general tier, so saving $100 while still getting a top-tier device makes the Galaxy S24 the more economical choice.',

## Task 5 — Structured Output & Error Handling

**Structured output:** the agent's final free-text answer is coerced into
a fixed Pydantic schema using `llm.with_structured_output(...)` — this is
LangChain's wrapper around the same "force the model to return this exact
JSON shape" feature the underlying Gemini/Anthropic APIs expose natively.

**Error handling:** `get_product_price` and `get_weather` above already
raise `ToolException` for missing data (same idea as Day 1's caught
exceptions). The piece that had to be *configured* to make that recovery
graceful is `handle_tool_error=True` on `AgentExecutor` (set in Task 3) —
without it, a raised `ToolException` would propagate up and crash
`.invoke()` instead of being caught and turned into an observation the
model can react to.

In [12]:
from pydantic import BaseModel, Field

class ProductRecommendation(BaseModel):
    product: str = Field(description="The recommended product name")
    price_usd: float = Field(description="The product's price in USD")
    recommendation: str = Field(description="One-line recommendation verdict")
    reasoning: str = Field(description="1-2 sentence justification")

structured_llm = llm.with_structured_output(ProductRecommendation)

structuring_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract a structured product recommendation from the assistant's answer below."),
    ("human", "{agent_answer}"),
])

structuring_chain = structuring_prompt | structured_llm
structured_result = structuring_chain.invoke({"agent_answer": turn3_answer})

print(structured_result)
print("\nAs dict:", structured_result.model_dump())

C:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


product='Samsung Galaxy S24' price_usd=699.0 recommendation='Best value flagship smartphone for budget-conscious buyers.' reasoning='At $699, it is $100 cheaper than the iPhone 15 while still offering top-tier flagship performance, making it the more economical choice.'

As dict: {'product': 'Samsung Galaxy S24', 'price_usd': 699.0, 'recommendation': 'Best value flagship smartphone for budget-conscious buyers.', 'reasoning': 'At $699, it is $100 cheaper than the iPhone 15 while still offering top-tier flagship performance, making it the more economical choice.'}


In [13]:
# Demonstrate the configured graceful recovery: ask for a product NOT in the database
print("=== Tool error demo (handled gracefully via handle_tool_error=True) ===")
error_demo = agent_executor.invoke({"input": "What is the price of the OnePlus 12?"})
print("\nFINAL ANSWER:", error_demo["output"])

=== Tool error demo (handled gracefully via handle_tool_error=True) ===


> Entering new AgentExecutor chain...


C:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Invoking: `get_product_price` with `{'product_name': 'OnePlus 12'}`


'OnePlus 12' not found in product database. Available: ['iphone 15', 'samsung galaxy s24', 'google pixel 8']

C:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': "I'm sorry, but the OnePlus 12 is not in the product database. The available products are:\n- iPhone 15\n- Samsung Galaxy S24\n- Google Pixel 8", 'index': 0, 'extras': {'signature': 'El4KXAERTTIPwbh5Kxzjzihbh4d4AoOgAa/KCXPgvf5cFWvFt1/37BFi3l1NrzO8SsrZHosXT7DeRqnHgQuezMQJpFF6bRmx4xZftFE8/ZRA3ws8E2blDbTj/TBwXrUg'}}]

> Finished chain.

FINAL ANSWER: [{'type': 'text', 'text': "I'm sorry, but the OnePlus 12 is not in the product database. The available products are:\n- iPhone 15\n- Samsung Galaxy S24\n- Google Pixel 8", 'index': 0, 'extras': {'signature': 'El4KXAERTTIPwbh5Kxzjzihbh4d4AoOgAa/KCXPgvf5cFWvFt1/37BFi3l1NrzO8SsrZHosXT7DeRqnHgQuezMQJpFF6bRmx4xZftFE8/ZRA3ws8E2blDbTj/TBwXrUg'}}]


### LangChain vs. Day 1 — what got easier, what got leaky

LangChain made **structured output** and **graceful tool-error recovery**
genuinely easier — `with_structured_output(...)` and `handle_tool_error=True`
replace what would otherwise be hand-written `try/except` blocks and JSON
schema parsing, in one line of configuration each. It also made the
*shape* of the agent loop easier to assemble: `create_tool_calling_agent`
+ `AgentExecutor` wire up the model, tools, and prompt without writing the
while-loop by hand.

The leakiness showed up directly, not hypothetically: `RunnableWithMessageHistory`
— the textbook memory solution — **crashed on the second turn** because it
stores the model's raw output message, and Gemini's "thinking" content
blocks (signed metadata) don't survive being replayed back through
LangChain's message-coercion layer. Day 1's raw approach never had this
problem, because *we* controlled exactly what got stored in `history` —
plain strings, nothing else. That's the trade-off in one sentence: the
convenience API abstracted away *so much* of the message-passing plumbing
that when the underlying model's output format didn't match what the
abstraction expected, the failure surfaced two layers deep in LangChain's
internals instead of in code we'd written and could immediately reason
about — exactly the kind of "magic" Day 1 was designed to demystify.